# Create ML Training Dataset

Combine selected protein structures with ligand affinity data to create training pairs:
- **Input**: (protein_structure, ligand)
- **Output**: affinity (IC50, pIC50, etc.)

## Workflow
1. Load best structure per UniProt
2. Load affinity data for each UniProt
3. Extract structure features (sequence, pocket, 3D coords)
4. Match with ligand data (SMILES, descriptors)
5. Create training samples

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import gemmi
from rdkit import Chem
from rdkit.Chem import Descriptors

## Step 1: Load Selected Structures

In [2]:
structures_df

NameError: name 'structures_df' is not defined

In [ ]:
# Load the best structure per UniProt
structures_df = pd.read_csv('structure_selection_results/best_structures_per_uniprot.csv')

# Filter for valid structures only (handle both empty strings and NaN)
valid_structures = structures_df[
    (structures_df['error'].fillna('') == '') &
    (structures_df['quality_score'].astype(float) > 250)  # Minimum quality threshold
]

print(f"Total UniProts: {len(structures_df)}")
print(f"Valid structures: {len(valid_structures)}")
print(f"\nQuality score range: {valid_structures['quality_score'].astype(float).min():.1f} - {valid_structures['quality_score'].astype(float).max():.1f}")

valid_structures.head()

Total UniProts: 830
Valid structures: 0

Quality score range: nan - nan


,uniprot_id,pdb_id,cif_path,method,resolution,chain_to_uniprot,ligands,chosen_ligand,pocket_residue_count,pocket_missing_CA_count,pocket_completeness,quality_score,error


## Step 2: Example - Process One UniProt

Let's demonstrate the workflow with one example (O00141)

In [ ]:
# Select a test UniProt
test_uniprot = 'O00141'
structure_row = valid_structures[valid_structures['uniprot_id'] == test_uniprot].iloc[0]

print(f"UniProt: {structure_row['uniprot_id']}")
print(f"PDB: {structure_row['pdb_id']}")
print(f"Resolution: {structure_row['resolution']} Å")
print(f"Quality Score: {structure_row['quality_score']}")
print(f"\nCIF Path: {structure_row['cif_path']}")

IndexError: single positional indexer is out-of-bounds

In [ ]:
# Load the affinity data for this UniProt
affinity_path = f"../../plate-vs/VLS_benchmark/chembl_affinity/uniprot_{test_uniprot}/{test_uniprot}_chembl_activities_filtered.parquet"

# Use duckdb for robust reading
import duckdb
df_affinity = duckdb.read_parquet(affinity_path).df()

print(f"Affinity data: {len(df_affinity)} compounds")
print(f"\nColumns: {list(df_affinity.columns)}")
print(f"\nFirst few entries:")
df_affinity.head()

## Step 3: Extract Structure Features

In [ ]:
def extract_protein_sequence(cif_path, chain_id='A'):
    """Extract protein sequence from structure."""
    struct = gemmi.read_structure(cif_path)
    if len(struct) == 0:
        return None
    
    model = struct[0]
    for chain in model:
        if chain.name == chain_id:
            # Extract sequence
            seq = []
            for res in chain:
                if res.entity_type == gemmi.EntityType.Polymer:
                    # Convert 3-letter to 1-letter code
                    aa = gemmi.find_tabulated_residue(res.name)
                    if aa.is_amino_acid():
                        seq.append(aa.one_letter_code)
            return ''.join(seq)
    return None

def extract_pocket_residues(cif_path, ligand_info, radius=6.0):
    """Extract pocket residue positions and identities."""
    # Parse ligand info: "ANP@A:500(heavy=31)"
    parts = ligand_info.split('@')
    if len(parts) < 2:
        return None
    
    ligand_name = parts[0]
    chain_res = parts[1].split('(')[0]  # "A:500"
    chain_id, res_id = chain_res.split(':')
    
    struct = gemmi.read_structure(cif_path)
    if len(struct) == 0:
        return None
    
    model = struct[0]
    
    # Find ligand
    lig_res = None
    for chain in model:
        if chain.name == chain_id:
            for res in chain:
                if str(res.seqid) == res_id and res.name.strip() == ligand_name:
                    lig_res = res
                    break
        if lig_res:
            break
    
    if not lig_res:
        return None
    
    # Get ligand positions
    lig_positions = [a.pos for a in lig_res if a.element.name != "H"]
    
    # Find nearby residues
    ns = gemmi.NeighborSearch(model, struct.cell, radius)
    ns.populate(include_h=False)
    
    pocket_residues = []
    for lig_pos in lig_positions:
        marks = ns.find_atoms(lig_pos, "\0", radius=radius)
        for mark in marks:
            cra = mark.to_cra(model)
            res = cra.residue
            if res.entity_type == gemmi.EntityType.Polymer:
                res_info = {
                    'chain': cra.chain.name,
                    'name': res.name.strip(),
                    'seqid': str(res.seqid),
                    'position': [cra.atom.pos.x, cra.atom.pos.y, cra.atom.pos.z]
                }
                if res_info not in pocket_residues:
                    pocket_residues.append(res_info)
    
    return pocket_residues

# Extract features for test UniProt
print("Extracting protein sequence...")
sequence = extract_protein_sequence(structure_row['cif_path'])
print(f"Sequence length: {len(sequence) if sequence else 'N/A'}")
if sequence:
    print(f"First 50 residues: {sequence[:50]}...")

print("\nExtracting pocket residues...")
pocket = extract_pocket_residues(structure_row['cif_path'], structure_row['chosen_ligand'])
print(f"Pocket residues: {len(pocket) if pocket else 'N/A'}")
if pocket:
    print(f"First 5 residues: {pocket[:5]}")

## Step 4: Process Ligand Data

In [ ]:
def compute_ligand_features(smiles):
    """Compute basic molecular descriptors for a ligand."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            'smiles': smiles,
            'mol_weight': Descriptors.MolWt(mol),
            'logp': Descriptors.MolLogP(mol),
            'hbd': Descriptors.NumHDonors(mol),
            'hba': Descriptors.NumHAcceptors(mol),
            'rotatable_bonds': Descriptors.NumRotatableBonds(mol),
            'tpsa': Descriptors.TPSA(mol),
            'num_atoms': mol.GetNumAtoms(),
        }
        return features
    except:
        return None

# Check what affinity columns we have
affinity_cols = [col for col in df_affinity.columns if 'value' in col.lower() or 'ic50' in col.lower() or 'pic50' in col.lower()]
print(f"Affinity columns: {affinity_cols}")

# Check for SMILES
smiles_col = None
for col in df_affinity.columns:
    if 'smiles' in col.lower():
        smiles_col = col
        break

print(f"SMILES column: {smiles_col}")

if smiles_col:
    # Compute features for first few ligands
    sample_smiles = df_affinity[smiles_col].dropna().head(3)
    print("\nExample ligand features:")
    for smiles in sample_smiles:
        features = compute_ligand_features(smiles)
        if features:
            print(f"\nSMILES: {smiles[:60]}...")
            print(f"  MW: {features['mol_weight']:.1f}")
            print(f"  LogP: {features['logp']:.2f}")
            print(f"  HBD: {features['hbd']}, HBA: {features['hba']}")

## Step 5: Create Training Samples

Combine protein structure with ligand data

In [ ]:
def create_training_samples(uniprot_id, structure_info, affinity_df):
    """
    Create training samples for one UniProt.
    Each sample: (protein_features, ligand_features, affinity)
    """
    samples = []
    
    # Extract protein features once
    cif_path = structure_info['cif_path']
    sequence = extract_protein_sequence(cif_path)
    pocket = extract_pocket_residues(cif_path, structure_info['chosen_ligand'])
    
    if not sequence or not pocket:
        print(f"Warning: Could not extract features for {uniprot_id}")
        return []
    
    protein_features = {
        'uniprot_id': uniprot_id,
        'pdb_id': structure_info['pdb_id'],
        'sequence': sequence,
        'sequence_length': len(sequence),
        'pocket_residues': len(pocket),
        'resolution': structure_info['resolution'],
        'quality_score': structure_info['quality_score'],
    }
    
    # Find SMILES and affinity columns
    smiles_col = None
    affinity_col = None
    
    for col in affinity_df.columns:
        if 'smiles' in col.lower() and smiles_col is None:
            smiles_col = col
        if 'standard_value' in col.lower() and affinity_col is None:
            affinity_col = col
    
    if not smiles_col or not affinity_col:
        print(f"Warning: Missing SMILES or affinity column for {uniprot_id}")
        return []
    
    # Create one sample per ligand
    for idx, row in affinity_df.iterrows():
        smiles = row[smiles_col]
        affinity = row[affinity_col]
        
        if pd.isna(smiles) or pd.isna(affinity):
            continue
        
        ligand_features = compute_ligand_features(smiles)
        if not ligand_features:
            continue
        
        # Combine everything
        sample = {
            **protein_features,
            **ligand_features,
            'affinity_value': affinity,
            'affinity_type': row.get('standard_type', 'IC50'),
        }
        samples.append(sample)
    
    return samples

# Test on our example
print(f"Creating training samples for {test_uniprot}...")
samples = create_training_samples(test_uniprot, structure_row, df_affinity)

print(f"\nCreated {len(samples)} training samples")

if samples:
    # Convert to DataFrame for easy viewing
    samples_df = pd.DataFrame(samples)
    print(f"\nSample data:")
    print(samples_df[['uniprot_id', 'pdb_id', 'sequence_length', 'mol_weight', 'logp', 'affinity_value']].head(10))

## Step 6: Process All UniProts

Now scale this to all UniProts

In [ ]:
def process_all_uniprots(structures_df, affinity_base_dir, output_file, max_uniprots=None):
    """
    Process all UniProts and create complete training dataset.
    """
    all_samples = []
    stats = {
        'total': 0,
        'success': 0,
        'no_affinity': 0,
        'extraction_failed': 0,
        'total_samples': 0,
    }
    
    # Filter valid structures
    valid = structures_df[
        (structures_df['error'] == '') &
        (structures_df['quality_score'].astype(float) > 250)
    ]
    
    if max_uniprots:
        valid = valid.head(max_uniprots)
    
    print(f"Processing {len(valid)} UniProts...\n")
    
    for idx, row in tqdm(valid.iterrows(), total=len(valid)):
        uniprot_id = row['uniprot_id']
        stats['total'] += 1
        
        # Load affinity data
        affinity_path = Path(affinity_base_dir) / f"uniprot_{uniprot_id}" / f"{uniprot_id}_chembl_activities_filtered.parquet"
        
        if not affinity_path.exists():
            stats['no_affinity'] += 1
            continue
        
        try:
            affinity_df = duckdb.read_parquet(str(affinity_path)).df()
            
            # Create samples
            samples = create_training_samples(uniprot_id, row, affinity_df)
            
            if samples:
                all_samples.extend(samples)
                stats['success'] += 1
                stats['total_samples'] += len(samples)
            else:
                stats['extraction_failed'] += 1
                
        except Exception as e:
            print(f"Error processing {uniprot_id}: {e}")
            stats['extraction_failed'] += 1
    
    # Save to file
    if all_samples:
        df_train = pd.DataFrame(all_samples)
        df_train.to_parquet(output_file, index=False)
        print(f"\n✓ Saved {len(df_train)} training samples to {output_file}")
    
    # Print stats
    print(f"\n" + "="*60)
    print("Processing Statistics:")
    print(f"  Total UniProts: {stats['total']}")
    print(f"  Success: {stats['success']} ({stats['success']/stats['total']*100:.1f}%)")
    print(f"  No affinity data: {stats['no_affinity']}")
    print(f"  Extraction failed: {stats['extraction_failed']}")
    print(f"  Total training samples: {stats['total_samples']:,}")
    if stats['success'] > 0:
        print(f"  Avg samples per UniProt: {stats['total_samples']/stats['success']:.1f}")
    
    return df_train if all_samples else None

# Test on first 5 UniProts
print("Testing on first 5 UniProts...\n")
df_train_test = process_all_uniprots(
    valid_structures,
    affinity_base_dir='../../plate-vs/VLS_benchmark/chembl_affinity',
    output_file='structure_selection_results/training_data_test.parquet',
    max_uniprots=5
)

In [ ]:
# Examine the test training data
if df_train_test is not None:
    print("Training data shape:", df_train_test.shape)
    print("\nColumns:", list(df_train_test.columns))
    print("\nFirst few samples:")
    display(df_train_test[[
        'uniprot_id', 'pdb_id', 'sequence_length', 'pocket_residues',
        'mol_weight', 'logp', 'affinity_value'
    ]].head(10))
    
    print("\nAffinity statistics:")
    print(df_train_test['affinity_value'].describe())

## Step 7: Run on Full Dataset

Once satisfied with the test, run on all UniProts

In [ ]:
# Uncomment to run on full dataset
# df_train_full = process_all_uniprots(
#     valid_structures,
#     affinity_base_dir='../../plate-vs/VLS_benchmark/chembl_affinity',
#     output_file='structure_selection_results/training_data_full.parquet',
#     max_uniprots=None  # Process all
# )

## Summary

This notebook demonstrates how to create ML training data by combining:
- **One representative structure per UniProt** (from structure selection)
- **Ligand affinity data** (from ChEMBL)

Each training sample contains:
- **Protein features**: sequence, pocket residues, structure quality
- **Ligand features**: SMILES, descriptors (MW, LogP, etc.)
- **Target**: affinity value (IC50, Ki, etc.)

### Next Steps:
1. **Feature engineering**: Add more sophisticated protein/ligand features
2. **Train/val/test split**: By UniProt or structure similarity
3. **Model training**: Graph neural networks, 3D CNNs, transformers
4. **Evaluation**: On held-out test set